# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_09 — Volatility Surface Construction

### Research question

How do we convert a noisy option chain into a clean, auditable implied-volatility surface that can be used for pricing, risk, calibration, and alpha research?

This notebook follows naturally from:

```text
02_04_implied_volatility_root_finding
```

That notebook solved implied volatility contract by contract. This notebook takes those contract-level IVs and builds a surface.

It covers:

1. synthetic option-chain generation;
2. quote cleaning and liquidity filters;
3. forward price and discount factor conventions;
4. conversion to log-moneyness;
5. total variance representation;
6. out-of-the-money option selection;
7. surface grids in $(k,T)$;
8. interpolation in implied volatility and total variance;
9. calendar-arbitrage diagnostics;
10. butterfly-arbitrage diagnostics;
11. simple surface repair by monotone total variance;
12. smile, skew, curvature, and term-structure features;
13. saved surface tables and validation reports.

The central idea is:

> A volatility surface is not just a pretty 3D plot. It is a cleaned, transformed, interpolated, and validated representation of option prices.

## 1. Why build a volatility surface?

An implied volatility surface maps strike and maturity to implied volatility:

$$
(K,T) \mapsto \sigma_{\text{imp}}(K,T)
$$
In practice, it is often more stable to work with:

$$
k = \log\left(\frac{K}{F_T}\right)
$$
where $F_T$ is the forward price, and with total implied variance:

$$
w(k,T)=\sigma_{\text{imp}}(k,T)^2T
$$
The surface is used for:

- marking option books;
- interpolating missing strikes and maturities;
- computing Greeks;
- calibrating stochastic-volatility models;
- detecting relative-value opportunities;
- estimating skew, curvature, and term structure;
- constructing volatility risk-premium features.

Bad surface construction can create artificial arbitrage, unstable Greeks, and misleading alpha signals.

## 2. Surface construction workflow

A robust workflow is:

1. ingest option chain;
2. validate bid/ask/mid prices;
3. compute forward and discount factors;
4. solve or import implied volatilities;
5. choose the most liquid quote representation;
6. convert strike to log-forward-moneyness;
7. convert IV to total variance;
8. filter unreliable quotes;
9. interpolate/smooth across strike and maturity;
10. check static arbitrage;
11. repair or flag violations;
12. save the surface with metadata and diagnostics.

This notebook uses a synthetic chain so it can run offline, but the same structure applies to real chains.

## 3. Static arbitrage intuition

A surface should not imply obvious static arbitrage.

### 3.1 Vertical spread monotonicity

For calls at fixed maturity, price should be decreasing in strike:

$$
C(K_1,T) \geq C(K_2,T)
\quad \text{for} \quad K_1<K_2
$$
For puts, price should be increasing in strike.

### 3.2 Butterfly convexity

Call prices should be convex in strike:

$$
\frac{\partial^2 C}{\partial K^2}\geq 0
$$
Discrete version: second finite differences should be non-negative on an equally spaced strike grid.

### 3.3 Calendar monotonicity

For the same strike, option value should generally not decrease as maturity increases under consistent forward and discount conventions.

In practice, one also checks total variance monotonicity in maturity at fixed log-moneyness:

$$
w(k,T_2) \geq w(k,T_1)
\quad \text{for} \quad T_2>T_1
$$
This is a practical diagnostic, not a complete proof of no arbitrage.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class SurfaceConfig:
    """
    Synthetic volatility surface construction configuration.
    """
    spot: float = 100.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.01
    seed: int = 42
    min_mid_price: float = 0.02
    max_relative_spread: float = 0.35
    min_vega: float = 0.02


config = SurfaceConfig()
config

## 5. BSM helpers

We need BSM prices and Vega to generate and validate the option chain.

The implied vol solver was built in the previous notebook, so here the main focus is the surface after IV extraction. Still, synthetic prices need a pricing function.

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal PDF.
    """
    x = np.asarray(x)
    return np.exp(-0.5 * x ** 2) / np.sqrt(2.0 * np.pi)


def bsm_d1_d2(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> tuple[float, float]:
    """
    BSM d1 and d2.
    """
    d1 = (
        log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * maturity
    ) / (volatility * sqrt(maturity))

    d2 = d1 - volatility * sqrt(maturity)

    return d1, d2


def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> float:
    """
    BSM European option price.
    """
    if maturity <= 0:
        if option_type == "call":
            return max(spot - strike, 0.0)
        if option_type == "put":
            return max(strike - spot, 0.0)
        raise ValueError("option_type must be 'call' or 'put'.")

    d1, d2 = bsm_d1_d2(
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        return float(discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2))

    if option_type == "put":
        return float(discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1))

    raise ValueError("option_type must be 'call' or 'put'.")


def bsm_vega(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float
) -> float:
    """
    BSM Vega.
    """
    d1, _ = bsm_d1_d2(
        spot=spot,
        strike=strike,
        maturity=maturity,
        risk_free_rate=risk_free_rate,
        dividend_yield=dividend_yield,
        volatility=volatility
    )

    return float(spot * exp(-dividend_yield * maturity) * normal_pdf(d1) * sqrt(maturity))

## 6. Synthetic true volatility surface

We create a synthetic volatility surface using forward log-moneyness:

$$
k=\log(K/F_T)
$$
The synthetic total structure contains:

- base level;
- skew;
- curvature/smile;
- short-maturity elevation;
- mild term-structure slope.

This gives us a controlled target surface.

In [ ]:
def forward_price(
    spot: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float
) -> float:
    """
    Forward price under continuous rates/dividends.
    """
    return spot * exp((risk_free_rate - dividend_yield) * maturity)


def discount_factor(
    maturity: float,
    risk_free_rate: float
) -> float:
    """
    Risk-free discount factor.
    """
    return exp(-risk_free_rate * maturity)


def true_synthetic_iv(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float
) -> float:
    """
    Synthetic implied volatility surface.

    Uses log-forward-moneyness k = log(K/F_T).
    """
    fwd = forward_price(spot, maturity, risk_free_rate, dividend_yield)
    k = log(strike / fwd)

    base = 0.17
    short_term_premium = 0.06 * exp(-3.0 * maturity)
    term_slope = 0.025 * (1 - exp(-1.5 * maturity))
    skew = -0.12 * k
    curvature = 0.55 * k ** 2
    mild_wing = 0.06 * abs(k) ** 1.5

    iv = base + short_term_premium + term_slope + skew + curvature + mild_wing

    return float(np.clip(iv, 0.05, 1.20))


# Quick check.
pd.Series({
    "ATM_1M_iv": true_synthetic_iv(config.spot, 100, 30/365, config.risk_free_rate, config.dividend_yield),
    "low_strike_1M_iv": true_synthetic_iv(config.spot, 80, 30/365, config.risk_free_rate, config.dividend_yield),
    "high_strike_1M_iv": true_synthetic_iv(config.spot, 120, 30/365, config.risk_free_rate, config.dividend_yield)
})

## 7. Generate synthetic option chain

The chain contains:

- maturity;
- strike;
- option type;
- theoretical price;
- noisy bid/ask;
- midpoint;
- synthetic implied volatility;
- Vega;
- forward;
- log-moneyness.

We also inject a small number of bad quotes to test the cleaning pipeline.

In [ ]:
def generate_synthetic_option_chain(
    config: SurfaceConfig,
    strikes: np.ndarray,
    maturities: np.ndarray,
    seed: int = 42
) -> pd.DataFrame:
    """
    Generate synthetic call/put option chain with bid/ask quotes.
    """
    local_rng = np.random.default_rng(seed)
    rows = []

    for T in maturities:
        F = forward_price(config.spot, T, config.risk_free_rate, config.dividend_yield)

        for K in strikes:
            true_iv = true_synthetic_iv(
                spot=config.spot,
                strike=float(K),
                maturity=float(T),
                risk_free_rate=config.risk_free_rate,
                dividend_yield=config.dividend_yield
            )

            for option_type in ["call", "put"]:
                price = bsm_price(
                    spot=config.spot,
                    strike=float(K),
                    maturity=float(T),
                    risk_free_rate=config.risk_free_rate,
                    dividend_yield=config.dividend_yield,
                    volatility=true_iv,
                    option_type=option_type
                )

                vega = bsm_vega(
                    spot=config.spot,
                    strike=float(K),
                    maturity=float(T),
                    risk_free_rate=config.risk_free_rate,
                    dividend_yield=config.dividend_yield,
                    volatility=true_iv
                )

                # Wider relative spreads for low-price / low-vega options.
                base_spread = max(0.01, 0.0025 * price + 0.05 / (1 + vega))
                noise = local_rng.normal(0.0, 0.10 * base_spread)

                mid = max(price + noise, 1e-6)
                bid = max(mid - base_spread / 2, 0.0)
                ask = mid + base_spread / 2

                rows.append({
                    "spot": config.spot,
                    "forward": F,
                    "discount_factor": discount_factor(T, config.risk_free_rate),
                    "strike": float(K),
                    "maturity": float(T),
                    "option_type": option_type,
                    "true_iv": true_iv,
                    "model_price": price,
                    "bid": bid,
                    "ask": ask,
                    "mid": 0.5 * (bid + ask),
                    "vega": vega,
                    "log_moneyness": log(float(K) / F)
                })

    chain = pd.DataFrame(rows)
    chain["bid_ask_spread"] = chain["ask"] - chain["bid"]
    chain["relative_spread"] = chain["bid_ask_spread"] / chain["mid"].clip(lower=1e-8)

    # In a real pipeline, this would be solved IV from 02_04.
    # Here we use noisy true IV as stand-in for recovered IV.
    chain["implied_vol"] = (
        chain["true_iv"]
        + local_rng.normal(0.0, 0.004, size=len(chain))
        + local_rng.normal(0.0, 0.002 / (1 + chain["vega"].to_numpy()), size=len(chain))
    ).clip(0.01, 3.0)

    return chain


strikes = np.arange(60, 141, 2)
maturities = np.array([7, 14, 30, 45, 60, 90, 120, 180, 270, 365, 540, 730]) / 365.0

raw_chain = generate_synthetic_option_chain(
    config=config,
    strikes=strikes,
    maturities=maturities,
    seed=config.seed
)

raw_chain.head()

In [ ]:
def inject_surface_quote_issues(chain: pd.DataFrame, seed: int = 2026) -> pd.DataFrame:
    """
    Inject bad quotes and IV outliers for pipeline testing.
    """
    local_rng = np.random.default_rng(seed)
    out = chain.copy()
    out["injected_issue"] = "none"

    # Cross a few markets.
    crossed_idx = local_rng.choice(out.index, size=8, replace=False)
    for idx in crossed_idx:
        old_bid = out.loc[idx, "bid"]
        old_ask = out.loc[idx, "ask"]
        out.loc[idx, "bid"] = old_ask
        out.loc[idx, "ask"] = old_bid
        out.loc[idx, "injected_issue"] = "crossed_market"

    # IV outliers.
    remaining = out.index.difference(crossed_idx)
    iv_outlier_idx = local_rng.choice(remaining, size=8, replace=False)
    out.loc[iv_outlier_idx, "implied_vol"] *= local_rng.choice([0.35, 2.5], size=len(iv_outlier_idx))
    out.loc[iv_outlier_idx, "injected_issue"] = "iv_outlier"

    # Stale zero-ish quotes.
    remaining = remaining.difference(iv_outlier_idx)
    stale_idx = local_rng.choice(remaining, size=6, replace=False)
    out.loc[stale_idx, "bid"] = 0.0
    out.loc[stale_idx, "ask"] = 0.001
    out.loc[stale_idx, "mid"] = 0.0005
    out.loc[stale_idx, "injected_issue"] = "stale_low_price"

    out["bid_ask_spread"] = out["ask"] - out["bid"]
    out["relative_spread"] = out["bid_ask_spread"] / out["mid"].clip(lower=1e-8)

    return out


chain = inject_surface_quote_issues(raw_chain, seed=2026)

chain["injected_issue"].value_counts()

## 8. Quote cleaning filters

A volatility surface should not use every listed quote blindly.

We filter using:

1. non-negative bid and ask;
2. ask greater than bid;
3. midpoint above minimum threshold;
4. relative spread below maximum threshold;
5. Vega above minimum threshold;
6. implied volatility within a plausible range.

These are not universal rules. They are research-pipeline defaults.

In [ ]:
def validate_surface_quotes(chain: pd.DataFrame, config: SurfaceConfig) -> pd.DataFrame:
    """
    Add quote validation flags and reasons.
    """
    out = chain.copy()

    conditions = {
        "non_negative_bid_ask": (out["bid"] >= 0) & (out["ask"] >= 0),
        "non_crossed_market": out["ask"] >= out["bid"],
        "mid_above_min": out["mid"] >= config.min_mid_price,
        "relative_spread_ok": out["relative_spread"] <= config.max_relative_spread,
        "vega_above_min": out["vega"] >= config.min_vega,
        "iv_plausible": out["implied_vol"].between(0.03, 2.0)
    }

    for name, mask in conditions.items():
        out[name] = mask

    out["quote_is_clean"] = np.logical_and.reduce(list(conditions.values()))

    reasons = []
    for _, row in out.iterrows():
        failed = [name for name in conditions if not row[name]]
        reasons.append(",".join(failed) if failed else "none")

    out["filter_reason"] = reasons

    return out


validated_chain = validate_surface_quotes(chain, config)

validated_chain["filter_reason"].value_counts().head(10)

In [ ]:
clean_chain = validated_chain[validated_chain["quote_is_clean"]].copy()

pd.Series({
    "raw_quotes": len(validated_chain),
    "clean_quotes": len(clean_chain),
    "dropped_quotes": len(validated_chain) - len(clean_chain),
    "clean_fraction": len(clean_chain) / len(validated_chain)
})

## 9. OTM quote selection

Market practitioners often use:

- OTM puts for low strikes;
- OTM calls for high strikes;
- near-ATM quote chosen by liquidity or midpoint quality.

This avoids using deep ITM options, which can be less liquid and more sensitive to funding/dividend assumptions.

Using forward moneyness:

$$
K < F_T \Rightarrow \text{use put}
$$
$$
K \geq F_T \Rightarrow \text{use call}
$$

In [ ]:
def select_otm_quotes(clean_chain: pd.DataFrame) -> pd.DataFrame:
    """
    Select OTM options for surface construction.
    """
    out = clean_chain.copy()

    out["is_otm"] = (
        ((out["strike"] < out["forward"]) & (out["option_type"] == "put"))
        | ((out["strike"] >= out["forward"]) & (out["option_type"] == "call"))
    )

    otm = out[out["is_otm"]].copy()
    otm["total_variance"] = otm["implied_vol"] ** 2 * otm["maturity"]
    otm["sqrt_total_variance"] = np.sqrt(otm["total_variance"])

    return otm.sort_values(["maturity", "log_moneyness"])


otm_quotes = select_otm_quotes(clean_chain)

otm_quotes.head()

In [ ]:
otm_summary = (
    otm_quotes
    .groupby("maturity")
    .agg(
        n_quotes=("implied_vol", "count"),
        min_k=("log_moneyness", "min"),
        max_k=("log_moneyness", "max"),
        min_iv=("implied_vol", "min"),
        max_iv=("implied_vol", "max")
    )
    .reset_index()
)

otm_summary.head()

## 10. Visualising raw smiles

Before interpolation, inspect raw OTM smiles by maturity.

This catches obvious quote problems and helps decide how much smoothing is appropriate.

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in otm_quotes.groupby("maturity"):
    if T in sorted(otm_quotes["maturity"].unique())[::2]:
        plt.plot(group["log_moneyness"], group["implied_vol"], marker="o", linestyle="-", label=f"T={T:.2f}")

plt.title("Raw OTM Implied Volatility Smiles")
plt.xlabel("Log-forward-moneyness k = log(K/F)")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in otm_quotes.groupby("maturity"):
    if T in sorted(otm_quotes["maturity"].unique())[::2]:
        plt.plot(group["log_moneyness"], group["total_variance"], marker="o", linestyle="-", label=f"T={T:.2f}")

plt.title("Raw OTM Total Variance Smiles")
plt.xlabel("Log-forward-moneyness k")
plt.ylabel("Total variance w = IV²T")
plt.legend()
plt.show()

## 11. Grid construction

We build a regular grid in:

$$
(k,T)
$$
where:

- $k$ is log-forward-moneyness;
- $T$ is maturity.

The surface grid makes later tasks easier:

- plotting;
- calibration;
- feature extraction;
- model comparison;
- interpolation to requested strikes and maturities.

In [ ]:
k_grid = np.linspace(-0.45, 0.45, 91)
T_grid = np.array([7, 14, 30, 45, 60, 90, 120, 180, 270, 365, 540, 730]) / 365.0

grid_points = pd.MultiIndex.from_product(
    [T_grid, k_grid],
    names=["maturity", "log_moneyness"]
).to_frame(index=False)

grid_points.head()

## 12. Slice-wise smile smoothing

A simple robust baseline is to fit each maturity smile separately.

For each maturity, we fit a quadratic model to total variance:

$$
w(k,T) \approx a(T)+b(T)k+c(T)k^2
$$
This is not a production volatility-surface model, but it is transparent and useful as a first surface-construction method.

More advanced methods include SVI, SABR, local volatility, Gaussian process smoothing, and arbitrage-constrained splines.

In [ ]:
def weighted_quadratic_fit_by_maturity(
    otm_quotes: pd.DataFrame,
    k_grid: np.ndarray,
    min_quotes: int = 5
) -> pd.DataFrame:
    """
    Fit maturity-by-maturity quadratic total variance smiles.

    Weights are based on Vega and inverse relative spread.
    """
    rows = []
    fit_coeffs = []

    for T, group in otm_quotes.groupby("maturity"):
        g = group.sort_values("log_moneyness").copy()

        if len(g) < min_quotes:
            continue

        k = g["log_moneyness"].to_numpy()
        w = g["total_variance"].to_numpy()

        weights = (
            g["vega"].to_numpy()
            / g["relative_spread"].clip(lower=0.01).to_numpy()
        )
        weights = weights / np.nanmax(weights)

        # Polynomial design: 1, k, k^2
        X = np.column_stack([np.ones_like(k), k, k ** 2])
        W = np.sqrt(weights)[:, None]

        beta, *_ = np.linalg.lstsq(X * W, w * W[:, 0], rcond=None)

        a, b, c = beta

        # Enforce non-negative curvature lightly.
        c_repaired = max(c, 1e-8)

        for kk in k_grid:
            fitted_w = a + b * kk + c_repaired * kk ** 2
            fitted_w = max(fitted_w, 1e-8)

            rows.append({
                "maturity": T,
                "log_moneyness": kk,
                "fitted_total_variance": fitted_w,
                "fitted_iv": sqrt(fitted_w / T)
            })

        fit_coeffs.append({
            "maturity": T,
            "a": a,
            "b": b,
            "c_raw": c,
            "c_used": c_repaired,
            "n_quotes": len(g),
            "mean_abs_residual": float(np.mean(np.abs(w - X @ beta)))
        })

    surface = pd.DataFrame(rows)
    coeffs = pd.DataFrame(fit_coeffs)

    return surface, coeffs


quadratic_surface, quadratic_coeffs = weighted_quadratic_fit_by_maturity(
    otm_quotes,
    k_grid=k_grid
)

quadratic_coeffs.head()

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in quadratic_surface.groupby("maturity"):
    if T in sorted(quadratic_surface["maturity"].unique())[::2]:
        plt.plot(group["log_moneyness"], group["fitted_iv"], label=f"T={T:.2f}")

plt.title("Quadratic-Fitted IV Surface by Maturity Slice")
plt.xlabel("Log-forward-moneyness")
plt.ylabel("Fitted implied volatility")
plt.legend()
plt.show()

## 13. Calendar monotonicity repair in total variance

A basic calendar diagnostic is:

$$
w(k,T_2)\geq w(k,T_1)
\quad \text{when} \quad T_2>T_1
$$
Noisy slice-wise fitting can violate this.

A simple repair is to apply a cumulative maximum in maturity for each $k$:

$$
\begin{aligned}
w_{\text{repaired}}(k,T_i) &= \max_{j\leq i}w_{\text{raw}}(k,T_j)
\end{aligned}
$$
This is crude but transparent.

Production systems generally use arbitrage-aware parameterisations or constrained optimisation instead.

In [ ]:
def repair_calendar_total_variance(surface: pd.DataFrame) -> pd.DataFrame:
    """
    Enforce nondecreasing total variance across maturity for each k.
    """
    out = surface.copy()
    out = out.sort_values(["log_moneyness", "maturity"])

    out["repaired_total_variance"] = (
        out
        .groupby("log_moneyness")["fitted_total_variance"]
        .cummax()
    )

    out["repaired_iv"] = np.sqrt(out["repaired_total_variance"] / out["maturity"])

    out["calendar_repair_amount"] = out["repaired_total_variance"] - out["fitted_total_variance"]

    return out.sort_values(["maturity", "log_moneyness"])


surface_repaired = repair_calendar_total_variance(quadratic_surface)

surface_repaired.head()

In [ ]:
pd.Series({
    "max_calendar_repair_amount": surface_repaired["calendar_repair_amount"].max(),
    "mean_calendar_repair_amount": surface_repaired["calendar_repair_amount"].mean(),
    "fraction_repaired_points": (surface_repaired["calendar_repair_amount"] > 1e-12).mean()
})

## 14. Surface heatmap

A volatility surface can be visualised as a heatmap.

We plot repaired implied volatility across log-moneyness and maturity.

In [ ]:
surface_pivot = surface_repaired.pivot(
    index="maturity",
    columns="log_moneyness",
    values="repaired_iv"
)

plt.figure(figsize=(12, 6))
plt.imshow(
    surface_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[
        surface_pivot.columns.min(),
        surface_pivot.columns.max(),
        surface_pivot.index.min(),
        surface_pivot.index.max()
    ]
)
plt.colorbar(label="Repaired implied volatility")
plt.title("Constructed Implied Volatility Surface")
plt.xlabel("Log-forward-moneyness")
plt.ylabel("Maturity")
plt.show()

## 15. Interpolation to arbitrary strike and maturity

A surface is useful because it can answer:

> What is the implied volatility at this strike and maturity?

We implement bilinear interpolation on the repaired regular grid.

For production, interpolation should be done carefully in total variance, not blindly in volatility.

In [ ]:
def bilinear_interpolate_surface(
    surface: pd.DataFrame,
    target_k: float,
    target_T: float,
    value_col: str = "repaired_total_variance"
) -> float:
    """
    Bilinear interpolation on a regular maturity/log-moneyness grid.
    """
    pivot = surface.pivot(index="maturity", columns="log_moneyness", values=value_col).sort_index()

    T_values = pivot.index.to_numpy()
    k_values = pivot.columns.to_numpy()

    if target_T < T_values.min() or target_T > T_values.max():
        raise ValueError("target_T outside surface maturity range.")

    if target_k < k_values.min() or target_k > k_values.max():
        raise ValueError("target_k outside surface log-moneyness range.")

    T_low = T_values[T_values <= target_T].max()
    T_high = T_values[T_values >= target_T].min()

    k_low = k_values[k_values <= target_k].max()
    k_high = k_values[k_values >= target_k].min()

    if T_low == T_high and k_low == k_high:
        return float(pivot.loc[T_low, k_low])

    if T_low == T_high:
        v_low = pivot.loc[T_low, k_low]
        v_high = pivot.loc[T_low, k_high]
        return float(v_low + (v_high - v_low) * (target_k - k_low) / (k_high - k_low))

    if k_low == k_high:
        v_low = pivot.loc[T_low, k_low]
        v_high = pivot.loc[T_high, k_low]
        return float(v_low + (v_high - v_low) * (target_T - T_low) / (T_high - T_low))

    Q11 = pivot.loc[T_low, k_low]
    Q12 = pivot.loc[T_high, k_low]
    Q21 = pivot.loc[T_low, k_high]
    Q22 = pivot.loc[T_high, k_high]

    weight_T = (target_T - T_low) / (T_high - T_low)
    weight_k = (target_k - k_low) / (k_high - k_low)

    interpolated = (
        Q11 * (1 - weight_k) * (1 - weight_T)
        + Q21 * weight_k * (1 - weight_T)
        + Q12 * (1 - weight_k) * weight_T
        + Q22 * weight_k * weight_T
    )

    return float(interpolated)


def surface_iv_at_strike_maturity(
    surface: pd.DataFrame,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float
) -> dict:
    """
    Interpolate repaired surface IV at strike and maturity.
    """
    F = forward_price(spot, maturity, risk_free_rate, dividend_yield)
    k = log(strike / F)

    total_variance = bilinear_interpolate_surface(
        surface,
        target_k=k,
        target_T=maturity,
        value_col="repaired_total_variance"
    )

    return {
        "strike": strike,
        "maturity": maturity,
        "forward": F,
        "log_moneyness": k,
        "total_variance": total_variance,
        "implied_vol": sqrt(total_variance / maturity)
    }


surface_query = surface_iv_at_strike_maturity(
    surface=surface_repaired,
    spot=config.spot,
    strike=103.0,
    maturity=75/365,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield
)

surface_query

## 16. Static arbitrage diagnostics from reconstructed call prices

To check surface quality, convert implied volatilities back into call prices on a strike grid.

At fixed maturity, call prices should be:

1. decreasing in strike;
2. convex in strike.

We use the repaired surface to generate call prices and then check these conditions.

In [ ]:
def build_call_price_grid_from_surface(
    surface: pd.DataFrame,
    spot: float,
    risk_free_rate: float,
    dividend_yield: float,
    strike_grid: np.ndarray,
    maturity_grid: np.ndarray
) -> pd.DataFrame:
    """
    Generate call prices from the constructed surface.
    """
    rows = []

    for T in maturity_grid:
        F = forward_price(spot, T, risk_free_rate, dividend_yield)

        for K in strike_grid:
            k = log(float(K) / F)

            # Skip outside interpolation domain.
            if k < surface["log_moneyness"].min() or k > surface["log_moneyness"].max():
                continue

            w = bilinear_interpolate_surface(
                surface,
                target_k=k,
                target_T=T,
                value_col="repaired_total_variance"
            )
            iv = sqrt(w / T)

            call = bsm_price(
                spot=spot,
                strike=float(K),
                maturity=float(T),
                risk_free_rate=risk_free_rate,
                dividend_yield=dividend_yield,
                volatility=iv,
                option_type="call"
            )

            rows.append({
                "maturity": float(T),
                "strike": float(K),
                "forward": F,
                "log_moneyness": k,
                "iv": iv,
                "total_variance": w,
                "call_price": call
            })

    return pd.DataFrame(rows)


surface_strike_grid = np.arange(65, 136, 1)
call_grid = build_call_price_grid_from_surface(
    surface=surface_repaired,
    spot=config.spot,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    strike_grid=surface_strike_grid,
    maturity_grid=T_grid
)

call_grid.head()

In [ ]:
def call_static_arbitrage_diagnostics(call_grid: pd.DataFrame) -> pd.DataFrame:
    """
    Check monotonicity and convexity of call prices by maturity.
    """
    rows = []

    for T, group in call_grid.groupby("maturity"):
        g = group.sort_values("strike").copy()
        prices = g["call_price"].to_numpy()
        strikes = g["strike"].to_numpy()

        first_diff = np.diff(prices)
        monotonicity_violations = int(np.sum(first_diff > 1e-8))

        # This assumes equally spaced strike grid.
        second_diff = prices[:-2] - 2 * prices[1:-1] + prices[2:]
        convexity_violations = int(np.sum(second_diff < -1e-6))

        rows.append({
            "maturity": T,
            "n_strikes": len(g),
            "monotonicity_violations": monotonicity_violations,
            "convexity_violations": convexity_violations,
            "min_first_difference": float(first_diff.min()) if len(first_diff) else np.nan,
            "min_second_difference": float(second_diff.min()) if len(second_diff) else np.nan
        })

    return pd.DataFrame(rows)


call_arb_report = call_static_arbitrage_diagnostics(call_grid)

call_arb_report

## 17. Calendar diagnostics

We check whether repaired total variance is nondecreasing in maturity for each log-moneyness grid point.

This is a direct diagnostic on the constructed surface grid.

In [ ]:
def calendar_diagnostics(surface: pd.DataFrame) -> pd.DataFrame:
    """
    Check total variance monotonicity across maturity for each log-moneyness.
    """
    rows = []

    for k, group in surface.groupby("log_moneyness"):
        g = group.sort_values("maturity")
        diffs = np.diff(g["repaired_total_variance"].to_numpy())

        rows.append({
            "log_moneyness": k,
            "calendar_violations": int(np.sum(diffs < -1e-12)),
            "min_total_variance_diff": float(diffs.min()) if len(diffs) else np.nan
        })

    return pd.DataFrame(rows)


calendar_report = calendar_diagnostics(surface_repaired)

pd.Series({
    "total_k_points": len(calendar_report),
    "k_points_with_calendar_violations": int((calendar_report["calendar_violations"] > 0).sum()),
    "total_calendar_violations": int(calendar_report["calendar_violations"].sum()),
    "min_total_variance_diff": calendar_report["min_total_variance_diff"].min()
})

## 18. Smile feature extraction

A volatility surface can become a feature set.

For each maturity, extract:

1. ATM implied volatility:

$$
\sigma_{\text{ATM}}(T)=\sigma(k=0,T)
$$
2. 25-delta-like skew proxy:

$$
\sigma(k=-0.10,T)-\sigma(k=0.10,T)
$$
3. curvature proxy:

$$
\frac{1}{2}\left[\sigma(k=-0.10,T)+\sigma(k=0.10,T)\right]-\sigma(k=0,T)
$$
These are simplified features, not exact delta-based market conventions.

In [ ]:
def extract_surface_features(surface: pd.DataFrame) -> pd.DataFrame:
    """
    Extract simple smile and term-structure features from repaired surface.
    """
    rows = []

    for T in sorted(surface["maturity"].unique()):
        def iv_at_k(k):
            w = bilinear_interpolate_surface(
                surface,
                target_k=k,
                target_T=T,
                value_col="repaired_total_variance"
            )
            return sqrt(w / T)

        iv_atm = iv_at_k(0.0)
        iv_left = iv_at_k(-0.10)
        iv_right = iv_at_k(0.10)
        iv_left_wing = iv_at_k(-0.25)
        iv_right_wing = iv_at_k(0.25)

        rows.append({
            "maturity": T,
            "atm_iv": iv_atm,
            "skew_10d_logmoneyness_proxy": iv_left - iv_right,
            "curvature_10d_logmoneyness_proxy": 0.5 * (iv_left + iv_right) - iv_atm,
            "left_wing_minus_atm": iv_left_wing - iv_atm,
            "right_wing_minus_atm": iv_right_wing - iv_atm
        })

    features = pd.DataFrame(rows)
    features["atm_iv_term_slope"] = features["atm_iv"].diff() / features["maturity"].diff()

    return features


surface_features = extract_surface_features(surface_repaired)

surface_features.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(surface_features["maturity"], surface_features["atm_iv"], marker="o", label="ATM IV")
plt.plot(surface_features["maturity"], surface_features["skew_10d_logmoneyness_proxy"], marker="o", label="Skew proxy")
plt.plot(surface_features["maturity"], surface_features["curvature_10d_logmoneyness_proxy"], marker="o", label="Curvature proxy")
plt.title("Surface Feature Term Structures")
plt.xlabel("Maturity")
plt.ylabel("Feature value")
plt.legend()
plt.show()

## 19. Comparing raw quotes to fitted surface

A surface should be close to reliable quotes but not overfit noisy points.

We compare raw OTM IVs with fitted surface IV at the same $(k,T)$.

In [ ]:
def compare_quotes_to_surface(
    otm_quotes: pd.DataFrame,
    surface: pd.DataFrame
) -> pd.DataFrame:
    """
    Compare raw OTM quotes with fitted surface at same k,T.
    """
    rows = []

    for row in otm_quotes.itertuples():
        k = float(row.log_moneyness)
        T = float(row.maturity)

        if (
            k < surface["log_moneyness"].min()
            or k > surface["log_moneyness"].max()
            or T < surface["maturity"].min()
            or T > surface["maturity"].max()
        ):
            continue

        w = bilinear_interpolate_surface(
            surface,
            target_k=k,
            target_T=T,
            value_col="repaired_total_variance"
        )

        fitted_iv = sqrt(w / T)

        rows.append({
            "maturity": T,
            "strike": row.strike,
            "log_moneyness": k,
            "option_type": row.option_type,
            "quote_iv": row.implied_vol,
            "surface_iv": fitted_iv,
            "iv_residual": fitted_iv - row.implied_vol,
            "abs_iv_residual": abs(fitted_iv - row.implied_vol),
            "vega": row.vega,
            "relative_spread": row.relative_spread
        })

    return pd.DataFrame(rows)


quote_fit_report = compare_quotes_to_surface(otm_quotes, surface_repaired)

quote_fit_report.head()

In [ ]:
fit_quality_summary = pd.Series({
    "mean_abs_iv_residual": quote_fit_report["abs_iv_residual"].mean(),
    "median_abs_iv_residual": quote_fit_report["abs_iv_residual"].median(),
    "p95_abs_iv_residual": quote_fit_report["abs_iv_residual"].quantile(0.95),
    "max_abs_iv_residual": quote_fit_report["abs_iv_residual"].max()
})

fit_quality_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(quote_fit_report["quote_iv"], quote_fit_report["surface_iv"], alpha=0.5)
min_v = min(quote_fit_report["quote_iv"].min(), quote_fit_report["surface_iv"].min())
max_v = max(quote_fit_report["quote_iv"].max(), quote_fit_report["surface_iv"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Quote IV vs Surface IV")
plt.xlabel("Quote IV")
plt.ylabel("Surface IV")
plt.show()

## 20. Optional SciPy interpolation

If SciPy is installed, we can compare the quadratic-slice method with unstructured interpolation.

This is optional. It is useful for visualisation but should be treated carefully:

- interpolation in IV can violate no-arbitrage;
- interpolation in total variance is usually safer;
- scattered-data interpolation can behave poorly outside dense regions.

The production approach should usually use constrained smoothing or an arbitrage-aware parameterisation.

In [ ]:
try:
    from scipy.interpolate import griddata
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SCIPY_AVAILABLE

In [ ]:
if SCIPY_AVAILABLE:
    points = otm_quotes[["maturity", "log_moneyness"]].to_numpy()
    values = otm_quotes["total_variance"].to_numpy()
    target_points = grid_points[["maturity", "log_moneyness"]].to_numpy()

    grid_w_linear = griddata(
        points=points,
        values=values,
        xi=target_points,
        method="linear"
    )

    scipy_surface = grid_points.copy()
    scipy_surface["linear_total_variance"] = grid_w_linear
    scipy_surface["linear_iv"] = np.sqrt(
        scipy_surface["linear_total_variance"] / scipy_surface["maturity"]
    )

    display(scipy_surface.head())
else:
    print("SciPy unavailable; optional griddata interpolation skipped.")

## 21. Saving surface outputs

The notebook saves:

1. raw synthetic chain;
2. validated chain;
3. clean OTM quotes;
4. quadratic fit coefficients;
5. repaired volatility surface;
6. call-price grid from surface;
7. static-arbitrage reports;
8. calendar report;
9. surface features;
10. quote-fit residual report;
11. manifest.

This makes the surface construction auditable.

In [ ]:
output_dir = Path("data/processed/volatility_surface_construction_v1")
output_dir.mkdir(parents=True, exist_ok=True)

raw_chain_path = output_dir / "raw_synthetic_option_chain.csv"
validated_chain_path = output_dir / "validated_option_chain.csv"
otm_quotes_path = output_dir / "clean_otm_quotes.csv"
coeffs_path = output_dir / "quadratic_slice_coefficients.csv"
surface_path = output_dir / "repaired_volatility_surface.csv"
call_grid_path = output_dir / "surface_call_price_grid.csv"
call_arb_path = output_dir / "call_static_arbitrage_report.csv"
calendar_report_path = output_dir / "calendar_arbitrage_report.csv"
features_path = output_dir / "surface_features.csv"
quote_fit_path = output_dir / "quote_fit_report.csv"
manifest_path = output_dir / "manifest.json"

chain.to_csv(raw_chain_path, index=False)
validated_chain.to_csv(validated_chain_path, index=False)
otm_quotes.to_csv(otm_quotes_path, index=False)
quadratic_coeffs.to_csv(coeffs_path, index=False)
surface_repaired.to_csv(surface_path, index=False)
call_grid.to_csv(call_grid_path, index=False)
call_arb_report.to_csv(call_arb_path, index=False)
calendar_report.to_csv(calendar_report_path, index=False)
surface_features.to_csv(features_path, index=False)
quote_fit_report.to_csv(quote_fit_path, index=False)

manifest = {
    "dataset_name": "volatility_surface_construction",
    "schema_version": "volatility_surface_construction_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_raw_quotes": int(len(chain)),
    "n_clean_quotes": int(len(clean_chain)),
    "n_otm_quotes": int(len(otm_quotes)),
    "k_grid_min": float(k_grid.min()),
    "k_grid_max": float(k_grid.max()),
    "n_k_grid": int(len(k_grid)),
    "n_maturities": int(len(T_grid)),
    "max_calendar_repair_amount": float(surface_repaired["calendar_repair_amount"].max()),
    "total_call_monotonicity_violations": int(call_arb_report["monotonicity_violations"].sum()),
    "total_call_convexity_violations": int(call_arb_report["convexity_violations"].sum()),
    "total_calendar_violations_after_repair": int(calendar_report["calendar_violations"].sum()),
    "mean_abs_iv_residual": float(fit_quality_summary["mean_abs_iv_residual"]),
    "notes": [
        "Surface uses OTM quotes selected by forward moneyness.",
        "Raw implied volatilities are generated synthetically as stand-ins for IVs solved in notebook 02_04.",
        "Surface is fitted in total variance by maturity-wise weighted quadratic regression.",
        "Calendar repair applies cumulative maximum to total variance across maturity.",
        "Static arbitrage diagnostics are simplified and should not be treated as a full production proof."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

surface_path, features_path, call_arb_path, calendar_report_path, manifest_path

## 22. Practical checklist for volatility surface construction

Before using a volatility surface, check:

1. **Quote quality**  
   Are bid/ask quotes valid, liquid, and non-stale?

2. **Contract metadata**  
   Are strikes, maturities, multipliers, exercise styles, and underlyings correct?

3. **Forward convention**  
   Is moneyness based on spot or forward?

4. **OTM selection**  
   Are you using liquid OTM calls and puts appropriately?

5. **Rates and dividends**  
   Are discount factors and forwards consistent?

6. **Total variance**  
   Are interpolation and smoothing performed in $w=\sigma^2T$ where appropriate?

7. **Calendar monotonicity**  
   Is total variance nondecreasing with maturity?

8. **Butterfly arbitrage**  
   Are call prices convex in strike?

9. **Vertical spreads**  
   Are call prices decreasing in strike and put prices increasing?

10. **Residuals**  
   Does the surface fit clean quotes without overfitting noisy outliers?

11. **Extrapolation**  
   Are wings and long maturities handled conservatively?

12. **Audit trail**  
   Can every filtered quote and surface repair be explained?

## 23. Limitations

### 23.1 Synthetic data

The notebook uses a synthetic option chain.

Real market data requires robust handling of stale quotes, wide spreads, corporate actions, dividends, futures forwards, and settlement conventions.

### 23.2 Simple quadratic slice model

The maturity-wise quadratic total-variance fit is transparent but not a production-quality smile model.

It can fail in wings or under strong skew.

### 23.3 Calendar repair is crude

Cumulative maximum repair enforces monotone total variance but may distort the surface.

A production system should use constrained optimisation or arbitrage-aware parameterisations.

### 23.4 Static arbitrage checks are incomplete

The notebook checks basic monotonicity, convexity, and total-variance calendar behaviour.

A full no-arbitrage validation is more subtle.

### 23.5 No bid/ask surface

The notebook constructs a mid-IV surface.

Real desks often maintain bid, mid, and ask volatility surfaces.

### 23.6 No dynamic surface modelling

This notebook builds one surface snapshot.

Surface dynamics require time-series modelling of level, skew, curvature, and term structure.

### 23.7 No model calibration yet

This surface can feed Heston, SABR, SVI, or local-volatility calibration, but calibration is not performed here.

## 24. Research frontier and current directions

Volatility surface construction remains an active area because market surfaces are noisy, high-dimensional, and constrained by arbitrage.

### 24.1 SVI and SSVI parameterisations

SVI models the total variance smile using a flexible parametric form.

SSVI extends the idea to surfaces and can be calibrated under conditions designed to avoid static arbitrage.

A future notebook could fit SVI slices and compare them with the quadratic baseline here.

### 24.2 Arbitrage-free smoothing

A production surface should avoid:

- calendar spread arbitrage;
- butterfly arbitrage;
- negative densities;
- inconsistent wings.

Methods include constrained splines, SVI/SSVI, convex optimisation, and arbitrage-penalised neural networks.

### 24.3 Local volatility

The Dupire local-volatility model extracts a local volatility function from the implied volatility surface.

This requires a smooth and arbitrage-consistent surface.

A future notebook could compute Dupire local volatility from this constructed surface.

### 24.4 Stochastic volatility calibration

Models such as Heston, SABR, and Bates are calibrated to implied volatility surfaces.

A future notebook could use the saved surface to fit Heston parameters and compare model-implied smiles.

### 24.5 Surface dynamics and alpha

Volatility surfaces move over time.

Features such as skew steepness, curvature, term-structure slope, and vol risk premium can become alpha or risk signals.

A future notebook could build daily surface features and test their predictive power.

### 24.6 Machine learning surfaces

Modern approaches use Gaussian processes, autoencoders, neural SDEs, and no-arbitrage neural networks to model surfaces.

The challenge is to combine flexibility with financial constraints.

### 24.7 Futures option surfaces

For futures options, Black-76 and futures forwards are usually more natural than spot Black-Scholes.

A later Chinese futures options notebook could adapt this pipeline to futures-implied volatility surfaces.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_11_heston_model_calibration**  
   Calibrating stochastic volatility to the constructed surface.

2. **02_12_sabr_smile_calibration**  
   Fitting SABR smiles by maturity.

3. **02_14_local_volatility_dupire_surface**  
   Computing Dupire local volatility from a smooth IV surface.

4. **03_09_volatility_surface_alpha_signals**  
   Turning surface level, skew, and curvature into predictive features.

5. **04_07_beta_weighted_tail_risk_hedging**  
   Using option surface information for crash-hedging decisions.

6. **05_03_transaction_costs_slippage_latency**  
   Connecting options marks to execution assumptions.

7. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone combining pricing, calibration, and volatility alpha.

## 26. Summary

This notebook built an implied volatility surface from a noisy option chain.

It showed how to:

1. generate a synthetic option chain;
2. validate bid/ask and IV quotes;
3. select OTM quotes;
4. convert strikes to log-forward-moneyness;
5. convert IV to total variance;
6. fit maturity-wise total-variance smiles;
7. repair calendar monotonicity;
8. interpolate the surface;
9. reconstruct call prices;
10. run static-arbitrage diagnostics;
11. extract skew, curvature, and term-structure features;
12. save all outputs and validation reports.

The key computational takeaway is:

> Surface construction is a data-engineering and numerical-analysis problem, not merely a plotting task.

The key financial takeaway is:

> A useful volatility surface must be clean, smooth enough to interpolate, and constrained enough not to imply obvious arbitrage.

## 27. Further reading

### Volatility surfaces

- Gatheral, J. *The Volatility Surface: A Practitioner's Guide.*
- Gatheral and Jacquier, "Arbitrage-free SVI volatility surfaces."
- Derman and Kani on local volatility.
- Dupire on local volatility.

### Surface models

- SVI and SSVI.
- SABR.
- Heston.
- Bates.
- Local volatility.
- Arbitrage-free splines.

### Numerical methods

- Interpolation in total variance.
- Constrained optimisation.
- Static arbitrage diagnostics.
- Surface smoothing.
- Gaussian processes and neural surfaces.

### Future extensions

- SVI smile fitting.
- Heston calibration.
- Dupire local volatility.
- Black-76 futures option surfaces.
- Surface alpha and volatility risk-premium features.